# DQ-07 · returns.csv
Checks: null rate, duplicates, FK, temporal (return_date ≥ order_date), business rules (refund ≤ payment, qty ≤ ordered).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
ret    = pd.read_csv('returns.csv', parse_dates=['return_date'])
orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
items  = pd.read_csv('order_items.csv', low_memory=False)
pay    = pd.read_csv('payments.csv')
prods  = pd.read_csv('products.csv')
print(f'Shape: {ret.shape}')
ret.head(3)

## 1. Null rate

In [ ]:
null_counts = ret.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(ret)*100).round(2)}))

## 2. Duplicate return_id

In [ ]:
flag('Duplicate return_id', ret.duplicated(subset='return_id'), ret)

## 3. FK: order_id → orders

In [ ]:
valid_orders = set(orders['order_id'])
flag('order_id not in orders', ~ret['order_id'].isin(valid_orders), ret)

## 4. FK: product_id → products

In [ ]:
valid_prods = set(prods['product_id'])
flag('product_id not in products', ~ret['product_id'].isin(valid_prods), ret)

## 5. Only 'returned' orders should have returns

In [ ]:
returned_orders = set(orders[orders['order_status']=='returned']['order_id'])
flag('Return for non-returned order status', ~ret['order_id'].isin(returned_orders), ret)

## 6. Domain values: return_reason

In [ ]:
VALID_REASON = {'wrong_size','defective','not_as_described','changed_mind','late_delivery'}
flag('Invalid return_reason', ~ret['return_reason'].isin(VALID_REASON), ret)

## 7. Business rule: return_quantity ≥ 1

In [ ]:
flag('return_quantity ≤ 0', ret['return_quantity'] <= 0, ret)

## 8. Business rule: return_quantity ≤ ordered quantity

In [ ]:
qty_ordered = items.groupby(['order_id','product_id'])['quantity'].sum().reset_index()
qty_ordered.columns = ['order_id','product_id','qty_ordered']
df = ret.merge(qty_ordered, on=['order_id','product_id'], how='left')
flag('return_quantity > ordered quantity', df['return_quantity'] > df['qty_ordered'], df)

## 9. Business rule: refund_amount ≤ payment_value

In [ ]:
df2 = ret.merge(pay[['order_id','payment_value']], on='order_id', how='left')
flag('refund_amount > payment_value', df2['refund_amount'] > df2['payment_value'], df2)
flag('refund_amount ≤ 0',            df2['refund_amount'] <= 0,                   df2)

## 10. Temporal: return_date ≥ order_date

In [ ]:
df3 = ret.merge(orders[['order_id','order_date']], on='order_id', how='left')
flag('return_date < order_date', df3['return_date'] < df3['order_date'], df3)

gap = (df3['return_date'] - df3['order_date']).dt.days
print(f'\nReturn gap (days) — order → return:')
print(gap.describe().round(1))
flag('Return gap > 365 days', gap > 365, df3)
flag('Return gap < 0 days',   gap < 0,   df3)

## Summary

In [ ]:
summary()